# Dashboard Migration: Apply Permissions (Modular)

## Overview

Apply permissions from source workspace to manually imported dashboards in target workspace using centralized configuration and helper modules.

**Prerequisites:**
1. ✅ Dashboards have been manually imported to target workspace
2. ✅ Permissions were captured during export (`notebooks/01_Export_and_Transform.ipynb`)
3. ✅ Configuration file updated (`config/config.yaml`)

**What this notebook does:**
1. Loads configuration from `config/config.yaml`
2. Connects to target workspace
3. Lists dashboards in target location
4. Matches with exported permissions by name/ID
5. Applies ACLs to each dashboard

**Features:**
- Dry-run mode for safe testing
- Flexible matching (by name, ID, or both)
- Works regardless of import method used

---

## Cell 0: Install Dependencies

In [ ]:
%pip install -U databricks-sdk pandas pyyaml --quiet
dbutils.library.restartPython()

## Cell 1: Import Helper Modules

In [ ]:
import sys
import pandas as pd

# Add helpers to path
sys.path.insert(0, '../helpers')

from helpers import (
    load_config,
    get_config,
    get_path,
    create_workspace_client,
    load_permissions_from_volume,
    list_target_dashboards,
    apply_dashboard_permissions
)

print("✅ Helper modules imported")

## Cell 2: Load Configuration

In [ ]:
print("="*80)
print("LOADING CONFIGURATION")
print("="*80)

# Load configuration from YAML file
config = load_config('../config/config.yaml')

# Extract key configuration
target_url = config['target']['workspace_url']
export_path = get_path('exported')
target_parent_path = config['paths']['target_parent_path']
match_by = config['permissions']['match_by']
dry_run = config['permissions']['dry_run']

print("\n✅ Configuration loaded\n")
print(f"   Target workspace: {target_url}")
print(f"   Looking for dashboards in: {target_parent_path}")
print(f"   Permissions source: {export_path}")
print(f"   Match by: {match_by}")
print(f"   Dry run: {dry_run}")

if dry_run:
    print("\n   ⚠️  DRY RUN MODE: Permissions will NOT be applied")
    print("      Set permissions.dry_run = false in config.yaml to apply")

## Cell 3: Load Permissions & Discover Dashboards

In [ ]:
print("\n" + "="*80)
print("LOADING PERMISSIONS AND DISCOVERING DASHBOARDS")
print("="*80)

# Connect to target workspace
print("\n🔗 Connecting to target workspace...")
target_client = create_workspace_client('target')
print(f"   ✅ Connected to: {target_url}")

# Load permissions from exported files
print(f"\n📋 Loading permissions from: {export_path}")
permissions_map = load_permissions_from_volume(export_path)
print(f"   ✅ Loaded permissions for {len(permissions_map)} dashboard(s)")

# List dashboards in target workspace
print(f"\n🔍 Discovering dashboards in: {target_parent_path}")
target_dashboards = list_target_dashboards(target_client, target_parent_path)
print(f"   ✅ Found {len(target_dashboards)} dashboard(s)")

# Display summary
if target_dashboards:
    df = pd.DataFrame(target_dashboards)
    display(df)
else:
    print("\n⚠️  No dashboards found in target workspace!")
    print(f"   Check that dashboards were imported to: {target_parent_path}")
    print("   Or try a broader path like '/' to see all dashboards")

## Cell 4: Match and Apply Permissions

In [ ]:
print("\n" + "="*80)
print("MATCHING AND APPLYING PERMISSIONS")
print("="*80)

if not target_dashboards:
    print("\n⚠️  No dashboards found in target workspace!")
    print(f"   Check that dashboards were imported to: {target_parent_path}")
elif not permissions_map:
    print("\n⚠️  No permissions found!")
    print(f"   Check that Notebook 01 captured permissions to: {export_path}")
else:
    results = []
    
    for dash in target_dashboards:
        dashboard_id = dash['id']
        dashboard_name = dash['name']
        
        print(f"\n📊 {dashboard_name}")
        print(f"   ID: {dashboard_id}")
        
        # Match permissions
        permissions_data = None
        matched_by = None
        
        if match_by in ["id", "both"]:
            permissions_data = permissions_map.get(dashboard_id)
            matched_by = "id"
        
        if not permissions_data and match_by in ["name", "both"]:
            permissions_data = permissions_map.get(dashboard_name)
            matched_by = "name"
        
        if not permissions_data:
            print(f"   ⚠️  No matching permissions found")
            results.append({
                'dashboard_name': dashboard_name,
                'status': 'no_match',
                'matched_by': None
            })
            continue
        
        print(f"   ✅ Matched permissions by: {matched_by}")
        
        # Apply permissions
        result = apply_dashboard_permissions(
            target_client,
            dashboard_id,
            permissions_data,
            dry_run=dry_run
        )
        
        if result['status'] == 'dry_run':
            print(f"   🔍 DRY RUN: Would apply {result['would_apply']} permission(s)")
            for perm in result['permissions']:
                principal = perm.get('user_name') or perm.get('group_name') or perm.get('service_principal_name')
                level = perm.get('all_permissions', ['unknown'])[0]
                print(f"      - {principal}: {level}")
        elif result['status'] == 'success':
            print(f"   ✅ Applied {result['applied']} permission(s)")
        elif result['status'] == 'failed':
            print(f"   ❌ Failed: {result['error']}")
        else:
            print(f"   ℹ️  {result.get('reason', 'Unknown status')}")
        
        results.append({
            'dashboard_name': dashboard_name,
            'dashboard_id': dashboard_id,
            'status': result['status'],
            'matched_by': matched_by
        })
    
    # Summary
    print("\n" + "="*80)
    print("PERMISSION APPLICATION SUMMARY")
    print("="*80)
    
    summary_df = pd.DataFrame([{
        'Dashboard': r['dashboard_name'],
        'Status': r['status'],
        'Matched By': r.get('matched_by', 'N/A')
    } for r in results])
    
    display(summary_df)
    
    if dry_run:
        print("\n⚠️  DRY RUN MODE - No permissions were actually applied")
        print("   Set permissions.dry_run = false in config/config.yaml to apply permissions")
    else:
        successful = len([r for r in results if r['status'] == 'success'])
        print(f"\n✅ Successfully applied permissions to {successful} dashboard(s)")
        print(f"\n🎉 Migration complete! Verify permissions in target workspace.")